## Week 2 Day 2

Our first Agentic Framework project!!

Prepare yourself for something ridiculously easy.

We're going to build a simple Agent system for generating cold sales outreach emails:
1. Agent workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs

## Before we start - some setup:


Please visit Sendgrid at: https://sendgrid.com/

(Sendgrid is a Twilio company for sending emails.)

If SendGrid gives you problems, see the alternative implementation using "Resend Email" in community_contributions/2_lab2_with_resend_email

Please set up an account - it's free! (at least, for me, right now).

Once you've created an account, click on:

Settings (left sidebar) >> API Keys >> Create API Key (button on top right)

Copy the key to the clipboard, then add a new line to your .env file:

`SENDGRID_API_KEY=xxxx`

And also, within SendGrid, go to:

Settings (left sidebar) >> Sender Authentication >> "Verify a Single Sender"  
and verify that your own email address is a real email address, so that SendGrid can send emails for you.


In [48]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio



In [49]:
load_dotenv(override=True)

True

In [27]:
# Let's just check emails are working for you

def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("caliguidpaul@gmail.com")  # Change to your verified sender
    to_email = To("caliguidpaul@gmail.com")  # Change to your recipient
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()

202


### Did you receive the test email

If you get a 202, then you're good to go!

#### Certificate error

If you get an error SSL: CERTIFICATE_VERIFY_FAILED then students Chris S and Oleksandr K have suggestions:  
First run this: `!uv pip install --upgrade certifi`  
Next, run this:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

#### Other errors or no email

If there are other problems, you'll need to check your API key and your verified sender email address in the SendGrid dashboard

Or use the alternative implementation using "Resend Email" in community_contributions/2_lab2_with_resend_email

(Or - you could always replace the email sending code below with a Pushover call, or something to simply write to a flat file)

## Step 1: Agent workflow

In [50]:
instructions1 = """
Role: Senior outbound sales specialist at ComplAI (AI-powered SOC 2 compliance SaaS).
Goal: Write a single cold email body that earns a positive reply or a 15–20 minute call.
Inputs: You may receive prospect context (company, role, pains, industry). If none, write a general version for a mid-market tech company.

Guidelines:
- Tone: professional, clear, credible; avoid hype/jargon.
- Personalization: use given details; do not fabricate. If none, stay neutral.
- Length: 120–160 words, 4–6 short paragraphs, skimmable.
- Value: tie ComplAI to SOC 2 speed, audit readiness, and reduced overhead; include 1 proof point only if provided.
- CTA: propose two time options and invite an alternative; one clear ask only.
- Compliance: no sensitive data, no unverifiable claims; be respectful and lawful.
- Formatting: plain text only; no 'Subject:' line; no markdown; minimal signature (name, company).

Output: Return only the email body ready to send.
"""

instructions2 = """
Role: Engaging outbound sales specialist at ComplAI (AI-powered SOC 2 compliance SaaS).
Goal: Write a single cold email body that feels personable and earns a reply.

Guidelines:
- Tone: warm, light, and tasteful; a single crisp, clever line is fine; never sarcastic or unprofessional.
- Personalization: use given details; do not invent specifics.
- Length: 100–140 words, short paragraphs.
- Clarity: emphasize how ComplAI reduces SOC 2 toil and smooths audits without buzzwords.
- CTA: invite a brief chat with two concrete time options; one clear ask.
- Formatting: plain text only; no 'Subject:' line; no emojis; minimal signature.

Output: Return only the email body ready to send.
"""

instructions3 = """
Role: Efficient outbound sales specialist at ComplAI (AI-powered SOC 2 compliance SaaS).
Goal: Write an ultra-concise cold email that earns a quick yes/no.

Guidelines:
- Tone: concise, direct, respectful; no fluff.
- Length: 60–90 words, 3–4 short paragraphs or a tight list.
- Focus: 1–2 concrete outcomes (faster SOC 2, audit readiness, less overhead).
- CTA: single-line ask with two time options; accept alternatives.
- Formatting: plain text only; no 'Subject:' line; minimal signature.

Output: Return only the email body ready to send.
"""


In [51]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-5"
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-5"
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-5"
)

In [6]:

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Error streaming response: Error code: 400 - {'error': {'message': 'Your organization must be verified to stream this model. Please go to: https://platform.openai.com/settings/organization/general and click on Verify Organization. If you just verified, it can take up to 15 minutes for access to propagate.', 'type': 'invalid_request_error', 'param': 'stream', 'code': 'unsupported_value'}}


BadRequestError: Error code: 400 - {'error': {'message': 'Your organization must be verified to stream this model. Please go to: https://platform.openai.com/settings/organization/general and click on Verify Organization. If you just verified, it can take up to 15 minutes for access to propagate.', 'type': 'invalid_request_error', 'param': 'stream', 'code': 'unsupported_value'}}

In [7]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Hello,

Reaching out because many mid-market teams spend a lot of time coordinating SOC 2 prep and audits.

ComplAI is an AI-assisted SOC 2 platform that connects to your cloud, identity, code, and ticketing tools to auto-collect evidence, map it to controls, flag gaps, and assign owners—so your audit trail stays current without spreadsheets.

You also get an auditor-ready workspace, policy management, risk and vendor tracking, and continuous monitoring—speeding up readiness and reducing overhead by keeping work in one place.

Open to a quick discussion about your SOC 2 plan? I can do Tuesday 10:00–10:20am PT or Thursday 2:00–2:20pm PT; if another time works better, I’m flexible.

Best,
Alex
ComplAI


Hi there,

SOC 2 shouldn’t feel like a second job. ComplAI trims the grind: it pulls evidence from your existing tools, maps it to controls, nudges owners for approvals, tracks vendors, and keeps policies audit-ready—so you spend minutes, not weeks, on prep.

Teams use us to see gaps earl

In [52]:
sales_picker = Agent(
    name="sales_picker",
    instructions="""
You are an impartial evaluator. You receive multiple cold email candidates and must select the single best one.
Selection criteria (in order): (1) clarity and credibility, (2) relevance to SOC 2 and audit readiness, (3) specificity and personalization, (4) brevity and readability, (5) a single, clear CTA.
If two are equally strong, pick the shorter one.
Return exactly the chosen email text and nothing else. Do not add commentary, labels, or quotes.
""",
    model="gpt-5"
)


In [9]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Best sales email:
SOC 2 can drain time from your team. I work with mid-market tech companies on a simpler path.

ComplAI’s AI-powered platform centralizes your controls, policies, and evidence so you can reach SOC 2 readiness faster and stay that way between audits.

We connect to your cloud, identity, code, and ticketing systems to gather evidence automatically, map controls, and track remediation—maintaining a clean audit trail without constant back-and-forth.

The result is less manual prep, fewer spreadsheets, and smoother auditor walkthroughs, so your engineers spend more time building and less time chasing screenshots.

Open to a quick 15–20 minute call to see if this could lighten your next audit cycle? I can do Tuesday 10:30am PT or Thursday 2:00pm PT—happy to work around your schedule if another time is better.

Best,
Alex
ComplAI


Now go and check out the trace:

https://platform.openai.com/traces

## Part 2: use of tools

Now we will add a tool to the mix.

Remember all that json boilerplate and the `handle_tool_calls()` function with the if logic..

In [53]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-5",
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-5",
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-5",
)

In [11]:
sales_agent1

Agent(name='Professional Sales Agent', instructions="\nRole: Senior outbound sales specialist at ComplAI (AI-powered SOC 2 compliance SaaS).\nGoal: Write a single cold email body that earns a positive reply or a 15–20 minute call.\nInputs: You may receive prospect context (company, role, pains, industry). If none, write a general version for a mid-market tech company.\n\nGuidelines:\n- Tone: professional, clear, credible; avoid hype/jargon.\n- Personalization: use given details; do not fabricate. If none, stay neutral.\n- Length: 120–160 words, 4–6 short paragraphs, skimmable.\n- Value: tie ComplAI to SOC 2 speed, audit readiness, and reduced overhead; include 1 proof point only if provided.\n- CTA: propose two time options and invite an alternative; one clear ask only.\n- Compliance: no sensitive data, no unverifiable claims; be respectful and lawful.\n- Formatting: plain text only; no 'Subject:' line; no markdown; minimal signature (name, company).\n\nOutput: Return only the email bo

## Steps 2 and 3: Tools and Agent interactions

Remember all that boilerplate json?

Simply wrap your function with the decorator `@function_tool`

In [54]:
@function_tool
def send_email(body: str):
    """ Send out an email with the given body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("caliguidpaul@gmail.com")  # Change to your verified sender
    to_email = To("caliguidpaul@gmail.com")  # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

### This has automatically been converted into a tool, with the boilerplate json created

In [55]:
# Let's look at it
send_email

FunctionTool(name='send_email', description='Send out an email with the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001E8927C7920>, strict_json_schema=True, is_enabled=True)

### And you can also convert an Agent into a tool

In [56]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001E8927B4040>, strict_json_schema=True, is_enabled=True)

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [57]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001E88FE13060>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001E892848400>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'

## And now it's time for our Sales Manager - our planning agent

In [16]:
# Improved instructions thanks to student Guillermo F.

instructions = """
You are the Sales Manager orchestrating three writer agents (sales_agent1/2/3) and one delivery tool (send_email). Your objective is to send exactly one high-quality cold email body.

Process:
1) Generate: Call all three sales_agent tools with the same brief to produce three distinct drafts. Do not write drafts yourself.
2) Evaluate: Compare drafts for clarity, relevance to SOC 2 and audit readiness, specificity/personalization, brevity, and a single, clear CTA. If tied, choose the shorter.
3) Send: Use send_email with the chosen draft as the body. Send exactly one email.

Constraints:
- Do not include a "Subject:" line in bodies you evaluate or send.
- Do not modify the chosen draft beyond removing any leading "Subject:" line, if present.
- Do not fabricate facts; prefer neutral language if information is missing.
- After calling send_email successfully, stop.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-5")

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Wait - you didn't get an email??</h2>
            <span style="color:#ff7800;">With much thanks to student Chris S. for describing his issue and fixes. 
            If you don't receive an email after running the prior cell, here are some things to check: <br/>
            First, check your Spam folder! Several students have missed that the emails arrived in Spam!<br/>Second, print(result) and see if you are receiving errors about SSL. 
            If you're receiving SSL errors, then please check out theses <a href="https://chatgpt.com/share/680620ec-3b30-8012-8c26-ca86693d0e3d">networking tips</a> and see the note in the next cell. Also look at the trace in OpenAI, and investigate on the SendGrid website, to hunt for clues. Let me know if I can help!
            </span>
        </td>
    </tr>
</table>

### And one more suggestion to send emails from student Oleksandr on Windows 11:

If you are getting certificate SSL errors, then:  
Run this in a terminal: `uv pip install --upgrade certifi`

Then run this code:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

Thank you Oleksandr!

## Remember to check the trace

https://platform.openai.com/traces

And then check your email!!


### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

With handoffs, control passes across



In [58]:

subject_instructions = """
You generate exactly one subject line for a cold sales email.
- 6–9 words; specific, clear, and credible.
- Avoid spammy terms (Free, Urgent, Limited time), ALL CAPS, and excessive punctuation.
- Personalize only with provided details; do not fabricate.
Return the subject text only, no quotes, no labels.
"""

html_instructions = """
Convert a plain-text sales email body into semantic, mobile-friendly HTML suitable for major email clients.
- Preserve content; do not add a subject or footer.
- Use <p>, <ul>, <li>, <a>; inline minimal, accessible styles; readable font sizing.
- Keep width fluid and layout simple; no external CSS or images.
- Ensure links are explicit and accessible.
Return only the HTML fragment for the email body.
"""

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-5")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-5")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [59]:
# Guardrails: validate email body before sending
from typing import List, Tuple
from agents import function_tool

def _word_count(s: str) -> int:
    return len([w for w in s.replace('\n',' ').split(' ') if w])

@function_tool
def validate_email_body(body: str, min_words: int = 50, max_words: int = 180) -> Dict[str, str]:
    """Ensure no 'Subject:' line and enforce word-count bounds.
    Returns {'status': 'ok'|'adjusted', 'body': '...', 'notes': '...'}
    """
    cleaned = body.strip()
    if cleaned.lower().startswith('subject:'):
        cleaned = '\n'.join(cleaned.splitlines()[1:]).lstrip()
    wc = _word_count(cleaned)
    notes = []
    if wc < min_words:
        notes.append(f"too short: {wc} words (<{min_words})")
    if wc > max_words:
        notes.append(f"too long: {wc} words (>{max_words})")
        # trim to max words
        tokens = [w for w in cleaned.split(' ') if w]
        cleaned = ' '.join(tokens[:max_words])
    status = 'ok' if not notes else 'adjusted'
    return { 'status': status, 'body': cleaned, 'notes': '; '.join(notes) }


In [60]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("caliguidpaul@gmail.com")  # Change to your verified sender
    to_email = To("caliguidpaul@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [61]:
tools = [subject_tool, html_tool, validate_email_body, send_html_email]


In [62]:
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001E8927C7380>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001E8911B0040>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='validate_email_body', description="Ensure no 'Subject:' line and enforce word-count bound

In [63]:
instructions = """
You are the Email Manager. Given a plain-text email body, you must:
1) Call subject_writer to generate a single strong subject (6–9 words, non-spammy).
2) Call validate_email_body to ensure no Subject: line and acceptable length.
3) Call html_converter to transform the same body into clean, mobile-friendly HTML.
4) Call send_html_email with the subject from (1) and HTML from (2).
Rules: Do not write the subject yourself; always use the tools in this order. Do not alter content other than HTML formatting. After send_html_email succeeds, stop.
"""

emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-5",
    handoff_description="Convert an email to HTML and send it")


### Now we have 3 tools and 1 handoff

In [64]:
# Mail merge sender (dry run by default)
from typing import List, Dict
import os, sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
from agents import function_tool

@function_tool
def send_bulk_html_email(recipients: List[str], subject: str, html_body: str, dry_run: bool = True, max_recipients: int = 10) -> Dict[str, str]:
    """Send the same HTML email to multiple recipients (guarded).
    Set dry_run=True to print instead of send. Hard-caps recipients to avoid mistakes.
    """
    if len(recipients) > max_recipients:
        return { 'status': 'blocked', 'reason': f'exceeds max_recipients ({max_recipients})' }
    if dry_run:
        return { 'status': 'dry_run', 'count': str(len(recipients)) }
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email(os.environ.get('SENDGRID_FROM', 'caliguidpaul@gmail.com'))
    for addr in recipients:
        to_email = To(addr)
        content = Content("text/html", html_body)
        mail = Mail(from_email, to_email, subject, content).get()
        sg.client.mail.send.post(request_body=mail)
    return { 'status': 'sent', 'count': str(len(recipients)) }


In [65]:
# Bulk Email Manager: validates, converts to HTML, and bulk sends (dry_run default)
bulk_instructions = """
You are the Bulk Email Manager. Given a plain-text email body and a list of recipients, you must:
1) Call subject_writer to generate a single strong subject (6–9 words, non-spammy).
2) Call validate_email_body to ensure no Subject: line and acceptable length; use the cleaned body.
3) Call html_converter to transform the body into clean, mobile-friendly HTML.
4) Call send_bulk_html_email with the subject and HTML. Default to dry_run=True unless explicitly told dry_run=false.
Stop after sending.
"""
bulk_emailer_agent = Agent(
    name="Bulk Email Manager",
    instructions=bulk_instructions,
    tools=[subject_tool, html_tool, validate_email_body, send_bulk_html_email],
    model="gpt-5",
    handoff_description="Format and bulk send an email to a list")


In [66]:
# Research tool: fetch URL text and a personalization agent
from typing import Dict
import requests
from bs4 import BeautifulSoup
from agents import Agent, Runner, function_tool

@function_tool
def fetch_url_text(url: str) -> Dict[str, str]:
    """Fetch and extract main text content from a web page.
    Returns { 'text': '...'}; keep inputs small to avoid long prompts.
    """
    resp = requests.get(url, timeout=15)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    # Remove script/style
    for tag in soup(['script','style','noscript']):
        tag.decompose()
    text = ' '.join(soup.get_text().split())
    return { 'text': text[:8000] }

personalizer_instructions = """
You research a prospect and write 2–3 short bullet points we can use to personalize a cold email.
- Use ONLY the provided fetched text; do not invent facts.
- Focus on recent initiatives, products, metrics, or pains that map to SOC 2 or audit readiness.
- Each bullet: <= 18 words, specific and verifiable.
- If the text is generic, return neutral, safe bullets.
Return bullets only.
"""
personalizer = Agent(name="Personalization Researcher", instructions=personalizer_instructions, tools=[fetch_url_text], model="gpt-5")


In [67]:
# Expose personalizer as a tool for orchestration
personalizer_tool = personalizer.as_tool(tool_name="personalizer", tool_description="Research a URL and produce 2–3 personalization bullets")


In [68]:
tools = [tool1, tool2, tool3, personalizer_tool]
handoffs = [emailer_agent, bulk_emailer_agent]
print(tools)
print(handoffs)


[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001E88FE13060>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x000001E892848400>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}

In [69]:
# Second Sales Manager (handoffs-oriented)
sales_manager2_instructions = """
You are the Sales Manager orchestrating three writer agents (sales_agent1/2/3) with optional research, and you hand off to the appropriate email manager to deliver.

Process:
0) Optional research: If the message contains a line like "research_url: https://...", call the personalizer tool with that URL and prepend the returned bullets (verbatim) to the brief you send to writers.
1) Generate: Call all three sales_agent tools with the same brief to produce three distinct drafts. Do not write drafts yourself.
2) Evaluate: Compare drafts for clarity, SOC 2 relevance, specificity/personalization, brevity, and a single, clear CTA. If tied, choose the shorter.
3) Handoff: If the message contains a line like "recipients: a@x.com,b@y.com", hand off to Bulk Email Manager and include the recipients; otherwise hand off to Email Manager.
4) Stop: Do not send directly yourself; rely on the Email Manager/Bulk Email Manager to deliver.

Constraints:
- Do not include a 'Subject:' line in bodies you evaluate or pass.
- Do not modify the chosen draft beyond removing any leading 'Subject:' line, if present.
- Do not fabricate facts; prefer neutral language if information is missing.
"""

sales_manager_2 = Agent(
    name="Sales Manager (handoffs)",
    instructions=sales_manager2_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-5",
)


In [72]:
# Run the second Sales Manager
example = """Send a cold sales email addressed to 'Dear CEO' from Alice
# Optional controls for the manager:
research_url: https://paulcaliguid.github.io/react-app/
recipients: caliguidpaul@gmail.com, pdcaliguid@gmail.com
dry_run=false
"""

with trace("Automated SDR (handoffs)"):
    result = await Runner.run(sales_manager_2, example)


### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Can you identify the Agentic design patterns that were used here?<br/>
            What is the 1 line that changed this from being an Agentic "workflow" to "agent" under Anthropic's definition?<br/>
            Try adding in more tools and Agents! You could have tools that handle the mail merge to send to a list.<br/><br/>
            HARD CHALLENGE: research how you can have SendGrid call a Callback webhook when a user replies to an email,
            Then have the SDR respond to keep the conversation going! This may require some "vibe coding" 😂
            </span>
        </td>
    </tr>
</table>

**Exercise Answers**
- **Design patterns:** Manager–worker orchestration (Sales Manager), Agents‑as‑tools, tool sequencing (Email Manager pipeline), handoffs (delegation to Email Manager), and parallel draft generation with asyncio.
- **Workflow → Agent (Anthropic):** Adding tools to an Agent so the model autonomously invokes them, e.g. `sales_manager = Agent(..., tools=tools, ...)`.
- **Extensions:** Add mail‑merge sender tool, a research/personalization agent, and stricter guardrails (length, tone, compliance).


**Hard Challenge: Handle SendGrid replies via Parse Webhook**
- **Configure:** In SendGrid, set up Inbound Parse for a subdomain (e.g., `reply.example.com`) to POST to your public URL (use ngrok in dev).
- **Receive:** Implement a webhook endpoint to accept multipart form (fields: `from`, `to`, `subject`, `text`, `html`).
- **Respond:** Pass the inbound text to an SDR agent to craft a short, helpful reply; send with a reply tool preserving the thread.
- **Security:** Lock down the endpoint (allowlist IPs, secret path, or a signature check), validate sender, and sanitize content.


In [ ]:
# --- deps you may need in your env (uncomment to install) ---
# !uv pip install fastapi uvicorn nest_asyncio python-multipart sendgrid python-dotenv

import os, re, threading, nest_asyncio, uvicorn, json
from typing import Dict, Optional

from fastapi import FastAPI, Request, Header, HTTPException
from fastapi.responses import JSONResponse
from dotenv import load_dotenv

import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content

from agents import Agent, Runner, function_tool

# -------------------------
# Env & Config
# -------------------------
load_dotenv(override=True)

SENDGRID_API_KEY = os.getenv("SENDGRID_API_KEY", "")
SENDGRID_FROM = os.getenv("SENDGRID_FROM", "caliguidpaul@gmail.com")  # must be verified in SendGrid
PARSE_SECRET = os.getenv("PARSE_SECRET")  # optional: set to any string to require a shared secret header
DRY_RUN = os.getenv("DRY_RUN", "true").lower() == "true"  # default safe: no actual sends

if not SENDGRID_API_KEY:
    print("⚠️  Missing SENDGRID_API_KEY in environment. Replies will fail unless you set it.")

# -------------------------
# Helper: reply sender
# -------------------------
@function_tool
def send_reply_email(to_email: str, subject: str, body: str) -> Dict[str, str]:
    """
    Send a reply email to the given address (plain text).
    Honors DRY_RUN env to avoid accidental sends.
    """
    if DRY_RUN:
        # Safe-path: don't actually send while testing
        print(f"🧪 DRY_RUN: would send -> to={to_email} subj={subject}\n{body}\n")
        return {"status": "dry_run", "to": to_email, "subject": subject}

    try:
        sg = sendgrid.SendGridAPIClient(api_key=SENDGRID_API_KEY)
        from_email = Email(SENDGRID_FROM)
        to_email_obj = To(to_email)
        content = Content("text/plain", body)
        mail = Mail(from_email, to_email_obj, subject, content).get()
        resp = sg.client.mail.send.post(request_body=mail)
        ok = 200 <= resp.status_code < 300
        print(f"📤 SendGrid response: {resp.status_code}")
        return {"status": "sent" if ok else "error", "code": resp.status_code}
    except Exception as e:
        print(f"❌ Send error: {e}")
        return {"status": "error", "reason": str(e)}

# -------------------------
# SDR Agent
# -------------------------
sdr_instructions = """
You are an SDR continuing an email thread. Read the inbound message and write a concise, helpful reply that advances the conversation.
- Tone: professional, friendly, and helpful.
- Length: <= 120 words.
- Focus: answer questions, propose the next step, and offer two time options; one clear CTA.
- No subject line; body only. Do not include signatures unless provided.
- Do not fabricate facts; ask a clarifying question if needed.
"""
sdr_responder = Agent(name="SDR Responder", instructions=sdr_instructions, model="gpt-5")

# -------------------------
# FastAPI app
# -------------------------
app = FastAPI()

@app.get("/health")
async def health():
    return {"ok": True, "dry_run": DRY_RUN}

def _extract_email(addr: str) -> str:
    """
    Pull an email address from 'Name <email@x.com>' or plain 'email@x.com'.
    """
    if not addr:
        return ""
    m = re.search(r"<\s*([^>]+)\s*>", addr)
    if m:
        return m.group(1).strip()
    return addr.strip()

def _threaded_subject(subject: str) -> str:
    """
    Ensure the subject is threaded with 'Re:' once.
    """
    if not subject:
        return "Re: Your email"
    if subject.lower().startswith("re:"):
        return subject
    return f"Re: {subject}"

@app.post("/sendgrid/inbound")
async def sendgrid_inbound(
    request: Request,
    x_parse_secret: Optional[str] = Header(default=None)  # SendGrid lets you add custom headers via your MTA; for dev, send with curl
):
    """
    Receive SendGrid Inbound Parse multipart/form-data.
    Expected fields: from, to, subject, text, html
    """

    # --- minimal shared-secret check (optional but recommended) ---
    if PARSE_SECRET and x_parse_secret != PARSE_SECRET:
        # Always return 200 to avoid SendGrid retries storm; indicate blocked.
        print("🔐 Blocked: bad PARSE_SECRET")
        return JSONResponse({"ok": False, "reason": "unauthorized"}, status_code=200)

    try:
        form = await request.form()
        # Raw dump for debugging (don’t log in prod)
        print("📥 Inbound form keys:", list(form.keys()))

        sender_raw = str(form.get("from") or "")
        to_raw = str(form.get("to") or "")
        subject_raw = str(form.get("subject") or "").strip()

        # Prefer 'text'; if missing but 'html' exists, strip tags roughly
        text_body = str(form.get("text") or "")
        html_body = str(form.get("html") or "")

        if not text_body and html_body:
            # crude HTML fallback
            text_body = re.sub(r"<[^>]+>", " ", html_body)
            text_body = re.sub(r"\s+", " ", text_body).strip()

        to_addr = _extract_email(sender_raw)  # reply back to the original sender
        reply_subject = _threaded_subject(subject_raw)

        if not to_addr:
            print("⚠️  No sender email found; cannot reply.")
            return {"ok": False, "reason": "missing sender"}

        # Generate AI reply
        print(f"🧠 Generating reply to: {to_addr} | subj: {reply_subject}")
        result = await Runner.run(sdr_responder, text_body or "(no body)")
        reply_body = result.final_output

        # Send reply (or dry-run)
        outcome = await send_reply_email.on_invoke_tool(
            ctx={},
            input=json.dumps({
                "to_email": to_addr,
                "subject": reply_subject,
                "body": reply_body
            })
        )

        # Always 200 to keep SendGrid happy; include outcome for debugging
        return {"ok": True, "outcome": outcome}

    except Exception as e:
        # Return 200 so SendGrid doesn’t spam retries, but include error
        print(f"❌ Webhook error: {e}")
        return {"ok": False, "error": str(e)}

# -------------------------
# Run inside Jupyter (non-blocking)
# -------------------------
nest_asyncio.apply()

def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

if not any(t.name == "uvicorn-bg" for t in threading.enumerate()):
    th = threading.Thread(target=_run, name="uvicorn-bg", daemon=True)
    th.start()
    print("✅ FastAPI running at http://127.0.0.1:8000   (GET /health)")
    print("   DRY_RUN =", DRY_RUN, "| Require PARSE_SECRET =", bool(PARSE_SECRET))
else:
    print("ℹ️  Server thread already running.")

✅ FastAPI running at http://127.0.0.1:8000   (GET /health)
   DRY_RUN = False | Require PARSE_SECRET = False


INFO:     Started server process [41428]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


📥 Inbound form keys: ['from', 'to', 'subject', 'text']
🧠 Generating reply to: caliguidpaul@gmail.com | subj: Re: Interested in SOC2
📤 SendGrid response: 202
INFO:     127.0.0.1:63877 - "POST /sendgrid/inbound HTTP/1.1" 200 OK
INFO:     127.0.0.1:54180 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:64872 - "GET / HTTP/1.1" 404 Not Found


In [2]:
import requests
r = requests.post(
    "http://127.0.0.1:8000/sendgrid/inbound",
    files={
        "from": (None, "Paul <caliguidpaul@gmail.com>"),
        "to": (None, "sales@yourapp.dev"),
        "subject": (None, "Interested in SOC2"),
        "text": (None, "Hi! Can we schedule a quick chat next week?"),
    },
    headers={"X-Parse-Secret": os.getenv("PARSE_SECRET", "")}  # if you set PARSE_SECRET
)
print(r.json())


{'ok': True, 'outcome': {'status': 'sent', 'code': 202}}


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>

## Extra note:

Google has released their Agent Development Kit (ADK). It's not yet got the traction of the other frameworks on this course, but it's getting some attention. It's interesting to note that it looks quite similar to OpenAI Agents SDK. To give you a preview, here's a peak at sample code from ADK:

```
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description="Agent to answer questions about the time and weather in a city.",
    instruction="You are a helpful agent who can answer user questions about the time and weather in a city.",
    tools=[get_weather, get_current_time]
)
```

Well, that looks familiar!

And a student has contributed a customer care agent in community_contributions that uses ADK.